---
# Loading data

In [ ]:
import pandas as pd

In [ ]:
def load_data() -> pd.DataFrame:
  return pd.read_excel("./data/task_2_data_ex.xlsx", sheet_name=0)

In [ ]:
df = load_data()

In [ ]:
df.head(15)

,year,month,produced_material,produced_material_production_type,produced_material_release_type,produced_material_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id
0,2024,1,10000,8002,FIN,990.0,50000,8002.0,PROD,990.0,RLT_10
1,2024,1,50000,8002,PROD,859.0,80070,8007.0,PROD,879.0,RLT_10
2,2024,1,50000,8002,PROD,859.0,90000,NaN,ADD,50.0,RLT_10
3,2024,1,50000,8002,PROD,859.0,90001,NaN,ADD,20.0,RLT_10
4,2024,1,80070,8007,PROD,929.0,80010,8001.0,PROD,3626.0,RLT_10
5,2024,1,80070,8007,PROD,929.0,90002,NaN,ADD,30.0,RLT_10
6,2024,1,80070,8007,PROD,929.0,90003,NaN,ADD,11.0,RLT_10
7,2024,1,80010,8001,PROD,1726.0,80000,8000.0,PROD,1818.0,RLT_10
8,2024,1,80010,8001,PROD,1726.0,90004,NaN,ADD,101.0,RLT_10
9,2024,1,80000,8000,PROD,1980.0,70000,NaN,RM,2668.0,RLT_10


---
# Analysis

We can derive the `Parent-Child` relationship, which is given by `Parent --> Child`. The production stages are:
```
High-purity polysilicon (8000) --> Single-crystal silicon ingot (8001)
--> Lapped silicon wafer (8007) --> Polished silicon wafer (8002),
```
so we can backtrack it from the `Polished silicon wafer` to `High-purity polysilicon`, in our case the relationship is:
`component_material --> produced_material`, so we will walk in the reverse direction: `produced_material --> component_material` and we will find the production steps.


---
# Implementation

## Create the `FIN` base

In [ ]:
def create_fin_base(df: pd.DataFrame) -> pd.DataFrame:
  df_fin = df[df['produced_material_release_type'] == 'FIN'].copy()
  df_fin.rename(columns={
        'produced_material': 'fin_material_id',
        'produced_material_release_type': 'fin_material_release_type',
        'produced_material_production_type': 'fin_material_production_type',
        'produced_material_quantity': 'fin_production_quantity',
  }, inplace=True)
  return df_fin

In [ ]:
df_fin = create_fin_base(df)
df_fin.head()

,year,month,fin_material_id,fin_material_production_type,fin_material_release_type,fin_production_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id
0,2024,1,10000,8002,FIN,990.0,50000,8002.0,PROD,990.0,RLT_10
11,2024,2,10000,8002,FIN,960.0,50000,8002.0,PROD,960.0,RLT_10
22,2024,3,10000,8002,FIN,968.0,50000,8002.0,PROD,968.0,RLT_10
33,2024,4,10000,8002,FIN,1006.0,50000,8002.0,PROD,1006.0,RLT_10
44,2024,5,10000,8002,FIN,1004.0,50000,8002.0,PROD,1004.0,RLT_10


## Create the hierarchy

In [ ]:
def create_hierarchy(df: pd.DataFrame, fin_base: pd.DataFrame) -> pd.DataFrame:
  return pd.merge(
      fin_base,
      df,
      left_on=['plant_id', 'year', 'component_material'],
      right_on=['plant_id', 'year', 'produced_material'],
      how='left',
      suffixes=('', '_step')
  )

In [ ]:
hierarchy = create_hierarchy(df, df_fin)
hierarchy.head()

,year,month,fin_material_id,fin_material_production_type,fin_material_release_type,fin_production_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id,month_step,produced_material,produced_material_production_type,produced_material_release_type,produced_material_quantity,component_material_step,component_material_production_type_step,component_material_release_type_step,component_material_quantity_step
0,2024,1,10000,8002,FIN,990.0,50000,8002.0,PROD,990.0,RLT_10,1,50000,8002,PROD,859.0,80070,8007.0,PROD,879.0
1,2024,1,10000,8002,FIN,990.0,50000,8002.0,PROD,990.0,RLT_10,1,50000,8002,PROD,859.0,90000,NaN,ADD,50.0
2,2024,1,10000,8002,FIN,990.0,50000,8002.0,PROD,990.0,RLT_10,1,50000,8002,PROD,859.0,90001,NaN,ADD,20.0
3,2024,1,10000,8002,FIN,990.0,50000,8002.0,PROD,990.0,RLT_10,2,50000,8002,PROD,839.0,80070,8007.0,PROD,980.0
4,2024,1,10000,8002,FIN,990.0,50000,8002.0,PROD,990.0,RLT_10,2,50000,8002,PROD,839.0,90000,NaN,ADD,49.0


## Finalize

In [ ]:
def final_result(hierarchy: pd.DataFrame) -> pd.DataFrame:
  final = hierarchy[[
        'plant_id',
        'fin_material_id',
        'fin_material_release_type',
        'fin_material_production_type',
        'fin_production_quantity',

        'component_material',
        'component_material_release_type',
        'component_material_production_type',
        'component_material_quantity',

        'component_material_step',
        'component_material_release_type_step',
        'component_material_production_type_step',
        'component_material_quantity_step',
        'year']]
  return final.rename(columns={
        'plant_id': 'plant',
        'fin_material_id': 'fin_material_id',
        'fin_material_release_type': 'fin_material_release_type',
        'fin_material_production_type': 'fin_material_production_type',
        'fin_production_quantity': 'fin_production_quantity',

        'component_material': 'prod_material_id',
        'component_material_release_type': 'prod_material_release_type',
        'component_material_production_type': 'prod_material_production_type',
        'component_material_quantity': 'prod_material_production_quantity',

        'component_material_step': 'component_id',
        'component_material_release_type_step': 'component_material_release_type',
        'component_material_production_type_step': 'component_material_production_type',
        'component_material_quantity_step': 'component_consumption_quantity',
        'year': 'year'
    })

In [ ]:
final = final_result(hierarchy)
final

,plant,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,year
0,RLT_10,10000,FIN,8002,990.0,50000,PROD,8002.0,990.0,80070,PROD,8007.0,879.0,2024
1,RLT_10,10000,FIN,8002,990.0,50000,PROD,8002.0,990.0,90000,ADD,NaN,50.0,2024
2,RLT_10,10000,FIN,8002,990.0,50000,PROD,8002.0,990.0,90001,ADD,NaN,20.0,2024
3,RLT_10,10000,FIN,8002,990.0,50000,PROD,8002.0,990.0,80070,PROD,8007.0,980.0,2024
4,RLT_10,10000,FIN,8002,990.0,50000,PROD,8002.0,990.0,90000,ADD,NaN,49.0,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4315,RLT_14,10009,FIN,8002,1081.0,50009,PROD,8002.0,1081.0,90045,ADD,NaN,49.0,2024
4316,RLT_14,10009,FIN,8002,1081.0,50009,PROD,8002.0,1081.0,90046,ADD,NaN,20.0,2024
4317,RLT_14,10009,FIN,8002,1081.0,50009,PROD,8002.0,1081.0,80079,PROD,8007.0,857.0,2024
4318,RLT_14,10009,FIN,8002,1081.0,50009,PROD,8002.0,1081.0,90045,ADD,NaN,51.0,2024


---
# SQL Implementation

## Table schema

```sql
create table if not exists production (
    yr int, -- year
    mo int, -- month
    produced_material varchar(50),
    produced_material_production_type int,
    produced_material_release_type varchar(10),
    produced_material_quantity decimal(10, 2),
    component_material varchar(50),
    component_material_production_type int,
    component_material_release_type varchar(10),
    component_material_quantity decimal(10, 2),
    plant_id varchar(20)
);
```

## View

```sql
create view production_stages as (
  select
      t0.plant_id as plant,
      t0.yr,
      t0.produced_material as fin_material_id,
      t0.produced_material_release_type as fin_material_release_type,
      t0.produced_material_production_type as fin_material_production_type,
      t0.produced_material_quantity as fin_production_quantity,

      t1.produced_material as prod_material_id,
      t1.produced_material_release_type as prod_material_release_type,
      t1.produced_material_production_type as prod_material_production_type,
      t1.produced_material_quantity as prod_material_production_quantity,

      t1.component_material as component_id,
      t1.component_material_release_type as component_material_release_type,
      t1.component_material_production_type as component_material_production_type,
      t1.component_material_quantity as component_consumption_quantity

  from production t0
  left join production t1 on
      t0.component_material = t1.produced_material
      AND t0.plant_id = t1.plant_id
      AND t0.yr = t1.yr
      AND t0.mo = t1.mo
  where t0.produced_material_release_type = 'FIN'
);
```